In [2]:
## IMPORTS AND SETUP
# Imports
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import datetime
import zipfile
import pandas as pd
import logging
import warnings
import math
import tensorflow_datasets as tfds
import gc
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm.notebook import tqdm
from sklearn.utils.class_weight import compute_class_weight
import wandb
import datetime
from wandb.integration.keras import WandbMetricsLogger
from dotenv import load_dotenv

# Force GPU usage
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow is using GPU: ", tf.test.is_gpu_available())
print("Devices: ", tf.config.list_physical_devices())
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        pass

# Hide TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0=default, 1=info, 2=warning, 3=error
tf.get_logger().setLevel(logging.ERROR)
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
tf.debugging.set_log_device_placement(False)
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.FATAL)

2025-04-11 07:39:45.327236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-11 07:39:45.366441: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-11 07:39:45.380045: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-11 07:39:45.456553: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Num GPUs Available:  1
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
TensorFlow is using GPU:  True
Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1744357190.373489   38316 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744357190.432387   38316 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744357190.432448   38316 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744357190.437234   38316 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744357190.437306   38316 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [20]:
## PARAMETERS
seed = 123
data_format_fix = False # Set to True to fix data formats
data_visualization = False # Set to True to visualize data

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset/Dataset_Reducted_binary_Painting" # Path to the raw data folder
excluded_folders = ["Dataset Livrable 2"] # Folders to exclude from the dataset
batch_size = 1000 # Batch size for dataset loading
img_height = 128 # Image height for dataset loading
img_width = 128 # Image width for dataset loading

# Dataset split parameters
train_split = 0.8 # Proportion of the dataset to use for training
val_split = 0.1 # Proportion of the dataset to use for validation
test_split = 0.1 # Proportion of the dataset to use for testing

binary_classification = True # Set to True to use binary classification, False for multi-class classification

learning_rate= 0.001
num_epochs = 10

model_name = "CNN_Dropout_5_0.3_Binary"


In [4]:
## IMPORTS AND SETUP
# Imports
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import datetime
import zipfile
import pandas as pd
from PIL import Image
import tensorflow_datasets as tfds


load_dotenv('../.env')

WANDB_API_KEY = os.getenv("API_KEY")

# Forcer TensorFlow à utiliser le GPU
gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         for gpu in gpus:
#             tf.config.experimental.set_memory_growth(gpu, True)
#     except RuntimeError as e:
#         print(e)

    
# Tensorboard setup
# %load_ext tensorboard


wandb.login(key=WANDB_API_KEY , relogin=True) 
run = wandb.init(
    project="Leyanda",
    config={
        "name": model_name +"_"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S"),
        "model_name": model_name,
        "learning_rate": learning_rate,
        "epochs": num_epochs,
    }
)

log_dir = f"logs/fit/{model_name}/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
    write_graph=True,
    update_freq='epoch'
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: fabien-ribes (tom-antoine-cesi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
## DATA PREPARATION FUNCTIONS
# Data format fixes
def data_formats_fixes(raw_data_path):
    """
    Walks through a directory to detect and remove problematic image files.
    Removes:
    - Files that are not actually JPG format
    - Corrupted or unreadable images
    Converts:
    - Invalid shape files to RGB format
    Parameters:
    - raw_data_path: Path to the raw data folder.
    """
    print(f"--Starting data format fixes--")
    print(f"Checking directory: {raw_data_path}")
    if not os.path.isdir(raw_data_path):
        print(f"Directory {raw_data_path} doesn't exist")
        return
    else:
        print(f"Directory {raw_data_path} exists")

    stats = {
        "processed": 0,
        "wrong_format_removed": 0,
        "invalid_shape_converted": 0, # Includes grayscale
        "corrupted_removed": 0,
        "valid_images": 0
    }

    total_files = 0
    for root, dirs, files in os.walk(raw_data_path):
        for file in files:
            if os.path.splitext(file)[1].lower() == '.jpg':
                total_files += 1

    with tqdm(total=total_files, desc="Checking images") as pbar:
        for root, dirs, files in os.walk(raw_data_path):
            for file in files:
                file_path = os.path.join(root, file)
                _, extension = os.path.splitext(file)

                if extension.lower() != '.jpg':
                    continue

                stats["processed"] += 1
                pbar.update(1)

                try:
                    with open(file_path, 'rb') as f:
                        header = f.read(4)

                    if header[:2] != b'\xff\xd8':  # Not a valid JPEG header
                        os.remove(file_path)
                        stats["wrong_format_removed"] += 1
                        continue

                    try:
                        with Image.open(file_path) as img:
                            if img.mode != 'RGB':
                                img_rgb = img.convert('RGB')
                                img_rgb.save(file_path, 'JPEG', quality=95)
                                stats["invalid_shape_converted"] += 1
                                stats["valid_images"] += 1
                                continue

                            stats["valid_images"] += 1

                    except Exception as e:
                        os.remove(file_path)
                        stats["corrupted_removed"] += 1

                except Exception as e:
                    os.remove(file_path)
                    stats["corrupted_removed"] += 1

    print(f"\nSummary:")
    print(f"Files processed: {stats['processed']}")
    print(f"Wrong format files removed: {stats['wrong_format_removed']}")
    print(f"Invalid shape files converted: {stats['invalid_shape_converted']}")
    print(f"Corrupted files removed: {stats['corrupted_removed']}")
    print(f"Valid images remaining: {stats['valid_images']}")
    print(f"Data format fixes completed.")


# Dataset assembly
def dataset_assembly(raw_data_path, subfolders, batch_size, img_height, img_width, seed):
    """
    Assembles a dataset from a folder structure.
    Parameters:
    - raw_data_path: Path to the raw data folder.
    - subfolders: List of subfolders to include in the dataset.
    Returns:
    - dataset: A TensorFlow dataset object.
    """
    print(f"--Starting dataset assembly--")
    try:
        dataset = tf.keras.utils.image_dataset_from_directory(
            raw_data_path,
            labels="inferred",
            label_mode="int",
            class_names=subfolders,
            color_mode="rgb",
            batch_size=batch_size,
            image_size=(img_height, img_width),
            shuffle=True,
            seed=seed,
            validation_split=None,
            subset=None,
            interpolation="bilinear",
            follow_links=False
        )

        class_names = dataset.class_names
        print(f"Detected classes: {class_names}")

        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

        for images, labels in dataset.take(1):
            print(f"Images batch shape : {images.shape}")

        print(f"Dataset assembly completed.")
        return dataset, class_names

    except Exception as e:
        print(f"Error creating dataset: {e}")


# Dataset split
def dataset_split(dataset, train_split=0.8, val_split=0.1, batch_size=1000, seed=123):
    """
    Splits a dataset into training, validation, and test sets.
    Parameters:
    - dataset: The dataset to split.
    - train_split: Proportion of the dataset to use for training.
    - val_split: Proportion of the dataset to use for validation.
    - test_split: Proportion of the dataset to use for testing.
    Returns:
    - train_ds: Training dataset.
    - val_ds: Validation dataset.
    - test_ds: Test dataset.
    """
    print(f"--Starting dataset split--")
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()  # Faster than len(dataset)
    print(f"Dataset size: {dataset_size}")
    train_size = int(train_split * dataset_size)
    val_size = int(val_split * dataset_size)
    test_size = dataset_size - train_size - val_size

    print(f"Creating training set of size ~{train_size*batch_size}...")
    train_ds = dataset.take(train_size)
    remaining_ds = dataset.skip(train_size)
    print(f"Creating validation set of size ~{val_size*batch_size}...")
    val_ds = remaining_ds.take(val_size)
    print(f"Creating test set of size ~{test_size*batch_size}...")
    test_ds = remaining_ds.skip(val_size)

    print(f"Dataset split completed.")
    return train_ds, val_ds, test_ds

In [ ]:
## DATA PREPARATION WORKFLOW
# 1. Fix data formats (if necessary)
if data_format_fix:
    data_formats_fixes(raw_data_path=raw_data_path)
subfolders = [f for f in os.listdir(raw_data_path) if os.path.isdir(os.path.join(raw_data_path, f)) and f not in excluded_folders]

# 2. Assemble dataset
dataset, class_names = dataset_assembly(
    raw_data_path=raw_data_path,
    subfolders=subfolders,
    batch_size=batch_size,
    img_height=img_height,
    img_width=img_width,
    seed=seed,
)

# 3. Split dataset
train_ds, val_ds, test_ds = dataset_split(
    dataset=dataset,
    train_split=train_split,
    val_split=val_split,
    batch_size=batch_size,
    seed=seed
)

--Starting dataset assembly--
Found 4997 files belonging to 5 classes.
UZHUHZIUZUJZOIHJIOHEIHE
Transforming to binary classification (positive class: 'Painting')
Images batch shape: (1000, 128, 128, 3), Labels batch shape: (1000,)
Dataset assembly completed.
--Starting dataset split--
Dataset size: 5
Creating training set of size ~4000...
Creating validation set of size ~0...
Creating test set of size ~1000...
Dataset split completed.


2025-04-11 07:47:18.865486: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 268439552 bytes after encountering the first element of size 268439552 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2025-04-11 07:47:18.896019: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 268439456 bytes after encountering the first element of size 268439456 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
2025-04-11 07:47:18.896207: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 268439456 bytes after encountering the first element of size 268439456 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ra

In [ ]:
## DATA VISUALIZATION FUNCTIONS
# Sample visualization
def visualize_class_samples(dataset, class_names, samples_per_class=5):
    """
    Visualize random samples from each class in the dataset.
    Parameters:
    - dataset: TensorFlow dataset
    - class_names: List of class names
    - samples_per_class: Number of samples to display per class
    """
    plt.figure(figsize=(15, 10))

    class_samples = {class_name: [] for class_name in class_names}

    for images, labels in dataset:
        for i, label in enumerate(labels.numpy()):
            class_name = class_names[label]
            if len(class_samples[class_name]) < samples_per_class:
                class_samples[class_name].append(images[i].numpy().astype("uint8"))

        if all(len(samples) >= samples_per_class for samples in class_samples.values()):
            break

    for idx, class_name in enumerate(class_names):
        for i, sample in enumerate(class_samples[class_name]):
            plt.subplot(len(class_names), samples_per_class, idx * samples_per_class + i + 1)
            plt.imshow(sample)
            plt.axis('off')
            if i == 0:
                plt.title(class_name)

    plt.tight_layout()
    plt.show()

# Visualize class distribution
def visualize_class_distribution(dataset, class_names):
    """
    Visualize the distribution of classes in the dataset.
    Parameters:
    - dataset: TensorFlow dataset
    - class_names: List of class names
    """
    class_counts = {class_name: 0 for class_name in class_names}

    for _, labels in dataset:
        for label in labels.numpy():
            class_counts[class_names[label]] += 1

    plt.figure(figsize=(12, 6))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title('Class Distribution')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
## DATA VISUALIZATION WORKFLOW
if data_visualization:
    # 1. Visualize class samples
    visualize_class_samples(dataset=train_ds, class_names=class_names, samples_per_class=5)

    # 2. Visualize class distribution
    visualize_class_distribution(dataset=train_ds, class_names=class_names)

In [ ]:
# Vérifier le chemin et le contenu
print(f"Chemin absolu : {os.path.abspath(raw_data_path)}")
if not os.path.exists(raw_data_path):
    raise ValueError("Le chemin spécifié n'existe pas.")
else:
    print(f"Contenu du dossier principal : {os.listdir(raw_data_path)}")

# Exclure le dossier spécifié
subfolders = [f for f in os.listdir(raw_data_path) if os.path.isdir(os.path.join(raw_data_path, f)) and f != excluded_folder]
print(f"Sous-dossiers utilisés pour le dataset : {subfolders}")

# Créer un dataset optimisé à partir du répertoire
try:
    dataset = tf.keras.utils.image_dataset_from_directory(
        raw_data_path,
        labels="inferred",  # Les labels sont inférés à partir des noms des sous-dossiers
        label_mode="int",  # Labels sous forme d'entiers
        class_names=subfolders,  # Spécifier les classes à inclure
        color_mode="rgb",  # Images en RGB
        batch_size=batch_size,
        image_size=(img_height, img_width),  # Redimensionner les images
        shuffle=True,
        seed=123,
        validation_split=None,  # Pas de validation split ici
        subset=None,
        interpolation="bilinear",
        follow_links=False
    )

    # Afficher les classes détectées
    class_names = dataset.class_names
    print(f"Classes détectées : {class_names}")

    # Précharger les données pour une meilleure performance
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

    # Afficher un exemple
    for images, labels in dataset.take(1):
        print(f"Images batch shape : {images.shape}")
        print(f"Labels batch : {labels.numpy()}")

except Exception as e:
    print(f"Erreur lors de la création du dataset : {e}")

# Proportions des splits
train_split = 0.8
val_split = 0.1
test_split = 0.1

# Calculer les tailles
dataset_size = tf.data.experimental.cardinality(dataset).numpy()  # Plus rapide que len(dataset)
train_size = int(train_split * dataset_size)
val_size = int(val_split * dataset_size)

# Mélanger le dataset
# dataset = dataset.shuffle(buffer_size=500, seed=123)

# Diviser le dataset
train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size).take(val_size)
test_ds = dataset.skip(train_size + val_size)

# Ajouter prefetch pour optimiser les performances
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

Chemin absolu : /tf/projet/Dataset_Reducted
Contenu du dossier principal : ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
Sous-dossiers utilisés pour le dataset : ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
Found 8274 files belonging to 5 classes.
Classes détectées : ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
Images batch shape : (32, 180, 180, 3)
Labels batch : [0 2 0 0 4 0 3 1 0 2 2 4 2 2 4 0 0 4 0 2 1 0 3 4 2 2 4 1 0 1 4 1]


In [ ]:
def create_model(img_height=180, img_width=180, num_classes=5):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(img_height, img_width, 3)),
        tf.keras.layers.Rescaling(1./255),
        
        tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Dense(num_classes)
    ])
    
    model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    model.save(f'./models/{model_name}.keras', include_optimizer=False)
    
    return model


In [ ]:
model = create_model(img_height=img_height, img_width=img_width, num_classes=len(class_names))

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 180, 180, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 90, 90, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 90, 90, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 90, 90, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 45, 45, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 45, 45, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     3,965,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,989,285 (15.22 MB)

 Trainable params: 3,989,285 (15.22 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import wandb
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

class ConfusionMatrixCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_data, class_names):
        super().__init__()
        self.val_data = val_data
        self.class_names = class_names

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []

        # Collect true labels and predictions
        for images, labels in self.val_data:
            preds = self.model.predict(images)
            y_true.extend(labels.numpy())
            y_pred.extend(np.argmax(preds, axis=1))

        # Generate confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(6, 6))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=self.class_names)
        disp.plot(ax=ax, xticks_rotation=45)
        plt.title(f"Confusion Matrix - Epoch {epoch + 1}")
        plt.tight_layout()

        # Log the confusion matrix as an image in wandb
        wandb.log({f"Confusion Matrix (Epoch {epoch + 1})": wandb.Image(fig)}, step=epoch)
        plt.close(fig)

        # Log the raw confusion matrix data for interactive dashboards in wandb
        wandb.log({f"Confusion Matrix Data (Epoch {epoch + 1})": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.class_names
        )}, step=epoch)

In [ ]:
def create_callbacks(model_name="default_model", tensorboard=True, early_stopping=True, model_checkpoint=True, conf_matrix=False, val_data=None, class_names=None):
    log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    callbacks = []

    if tensorboard:
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
        callbacks.append(tensorboard_callback)

    if early_stopping:
        early_stopping_callback = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=4,
            restore_best_weights=True
        )
        callbacks.append(early_stopping_callback)

    if model_checkpoint:
        checkpoint_dir = "checkpoints"
        os.makedirs(checkpoint_dir, exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(checkpoint_dir, f"{model_name}_{timestamp}.keras"),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
        callbacks.append(model_checkpoint_callback)

    if conf_matrix and val_data is not None and class_names is not None:
        cm_callback = ConfusionMatrixCallback(val_data=val_data, class_names=class_names)
        callbacks.append(cm_callback)

    return callbacks

callbacks = [WandbMetricsLogger()] + create_callbacks(
    model_name=model_name,
    tensorboard=True,
    early_stopping=True,
    model_checkpoint=True,
    conf_matrix=True,  # Activer la matrice de confusion
    val_data=val_ds,  # Passer les données de validation
    class_names=class_names  # Passer les noms des classes
)

# Ajouter le callback de la matrice de confusion
conf_matrix_callback = ConfusionMatrixCallback(val_data=val_ds, class_names=class_names)
callbacks.append(conf_matrix_callback)

In [ ]:
from collections import Counter

def count_class_occurrences(dataset):
    label_counts = Counter()
    for images, labels in dataset:
        label_counts.update(labels.numpy())
    return dict(label_counts)

y_train = []

for _, labels in train_ds:
    y_train.extend(labels.numpy())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print("Class weights :", class_weights_dict)





Class weights : {0: 0.831638418079096, 1: 0.8368919772583702, 2: 0.8203095975232199, 3: 5.810526315789474, 4: 0.8254205607476636}


In [ ]:
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=num_epochs,
  callbacks=callbacks,
  class_weight=class_weights_dict
)

run.finish()

model.evaluate(test_ds)

In [75]:
import numpy as np

# Utiliser les vraies classes détectées dans le dataset
# Assurez-vous que `class_names` est défini à partir de votre dataset
# Exemple : class_names = dataset.class_names
print(f"Classes disponibles : {class_names}")

model = tf.keras.models.load_model('./models/'+model_name+'.keras')
model.load_weights('./checkpoints/CNN_Dropout_5_0.3_20250409-144535.keras')
# Charger l'image
img_path = 'peinture_vernon.jpg'  # Remplacez par le chemin de votre image
img = tf.keras.utils.load_img(img_path, target_size=(img_height, img_width)) 

# Prétraiter l'image
img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # Ajouter une dimension pour le batch

# Faire une prédiction
predictions = model.predict(img_array)


# Trouver l'indice de la classe avec la probabilité la plus élevée
predicted_class_index = np.argmax(predictions[0])

print(f"Prédictions : {predictions[0]}")

# Afficher le nom de la classe prédite
predicted_class_name = class_names[predicted_class_index]
print(f"Classe prédite : {predicted_class_name}")

Classes disponibles : ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
Prédictions : [ 4.8858256   4.1452675   0.40046328 -4.100996   -2.0577497 ]
Classe prédite : Painting
